In [1]:
from impisc.et_daqbox import daq_box_api as dba
from impisc.packets import RTDPacket
import pathlib
import ctypes
from matplotlib import pyplot as plt
from astropy import units as u
import astropy.time as atime
import numpy as np
import pathlib

%matplotlib qt
plt.style.use("nice.mplstyle")

ModuleNotFoundError: No module named 'impisc'

In [14]:
import lzma
direc = pathlib.Path("ba133-drift/temps")
sipm_temps = list()
times = list()
for filename in direc.iterdir():
    with lzma.open(filename, "rb") as file:
        while True:
            data = file.read(ctypes.sizeof(RTDPacket))
            if len(data) == 0:
                break
            packet = RTDPacket.from_buffer_copy(data)
            sipm_temps.append(packet.temperature6)
            times.append(packet.timestamp)


sipm_temps = sipm_temps << u.deg_C
times = atime.Time(times, format="unix")

In [16]:
import datetime

wf_direc = pathlib.Path("ba133-drift/waveforms/")

iter_per_sleep = 20
sleep_dur = datetime.timedelta(seconds=5)
all_waveforms = dict()

for fn in sorted(wf_direc.iterdir()):
    if not fn.name.endswith(".bin"):
        continue
    with open(fn, "rb") as f:
        start = datetime.datetime.strptime(
            fn.stem.split("_")[1],
            "%Y-%j-%H-%M-%S"
        )
        time_idx = 0
        while True:
            data = f.read(8000)
            if len(data) == 0:
                break
            wf = dba.parse_waveform_packet(data)["data"]

            cur_time = (time_idx // iter_per_sleep) * sleep_dur + start
            time_idx += 1
            if cur_time not in all_waveforms:
                all_waveforms[cur_time] = list()
            
            all_waveforms[cur_time].append(wf)

In [17]:
unique_times = list(all_waveforms)
avg_func = np.mean
medians = [
    np.mean(
        avg_func(np.array(all_waveforms[k])[:, 250:], axis=1)
    )
    for k in unique_times
]

stdevs = [
    np.std(
        avg_func(np.array(all_waveforms[k])[:, 250:], axis=1)
    )
    for k in unique_times
]

In [18]:
fig, (baseline_ax, temp_ax) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 6))
baseline_ax.errorbar(
    unique_times,
    medians,
    stdevs,
    marker='.',
    markersize=2,
    color='red',
    linestyle='none',
    elinewidth=1,
    capsize=4,
    alpha=0.5,
    ecolor='black'
)

temp_ax.scatter(
    times.datetime,
    sipm_temps.to_value(u.deg_C),
    s=2,
    color='blue'
)

temp_ax.set_xlim(
    datetime.datetime(2026, 5, 18, 17, 55),
    datetime.datetime(2026, 5, 18, 19, 40),
)

fig.suptitle("Baseline/temperature dependence with a recalibration in the middle")
temp_ax.set(xlabel='UTC', ylabel='SiPM temp (deg C)')
baseline_ax.set(ylabel='DAQBOX waveform baseline (mV)')

[Text(0, 0.5, 'DAQBOX waveform baseline (mV)')]